In [1]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler

print("\n1. Loading Data & Mapping Nodes...")
df = pd.read_csv('HI_Small_1M_Chronological.csv')

# Extract unique nodes and create mapping
senders = df['Account'].unique()
receivers = df['Account.1'].unique()
unique_accounts = np.unique(np.concatenate((senders, receivers)))
print(f"Total Unique Accounts (Nodes): {len(unique_accounts)}")

account_to_id = {account: i for i, account in enumerate(unique_accounts)}
df['Sender_Node_ID'] = df['Account'].map(account_to_id)
df['Receiver_Node_ID'] = df['Account.1'].map(account_to_id)

print("\n2. Engineering Node Features (The 'Whale Crusher' Update)...")
# Aggregate Outgoing
outgoing = df.groupby('Account').agg(
    total_sent=('Amount Paid', 'sum'),
    avg_sent=('Amount Paid', 'mean'),
    count_sent=('Amount Paid', 'count')
).reset_index().rename(columns={'Account': 'Node'})

# Aggregate Incoming
incoming = df.groupby('Account.1').agg(
    total_received=('Amount Received', 'sum'),
    avg_received=('Amount Received', 'mean'),
    count_received=('Amount Received', 'count')
).reset_index().rename(columns={'Account.1': 'Node'})

# Merge into complete profiles
node_profiles = pd.merge(outgoing, incoming, on='Node', how='outer').fillna(0)

# --- THE ADVANCED AML TRICKS ---
# Trick 1: The Mule Ratio (Money In vs. Money Out)
# We add +1 to prevent dividing by zero errors
node_profiles['flow_ratio'] = (node_profiles['total_received'] + 1) / (node_profiles['total_sent'] + 1)

# Trick 2: Log-Transform the Financial Amounts to crush the Whales
for col in ['total_sent', 'avg_sent', 'total_received', 'avg_received']:
    node_profiles[col] = np.log1p(node_profiles[col])

print("\n3. Aligning Features & Scaling for Neural Networks...")
ordered_nodes = pd.DataFrame({'Node': unique_accounts})
aligned_profiles = pd.merge(ordered_nodes, node_profiles, on='Node', how='left').fillna(0)

# We now have 7 highly-optimized features
features_array = aligned_profiles[['total_sent', 'avg_sent', 'count_sent', 
                                   'total_received', 'avg_received', 'count_received', 
                                   'flow_ratio']].values

# Standard scaling is now safe because the extreme outliers are crushed!
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_array)

x_tensor = torch.tensor(features_scaled, dtype=torch.float)
print(f"NEW Feature Matrix Shape: {x_tensor.shape} (Should be [{len(unique_accounts)}, 7])")

print("\n4. Constructing Edges & Weights...")
# Edge Index
source_nodes = df['Sender_Node_ID'].values
target_nodes = df['Receiver_Node_ID'].values
edge_index = torch.tensor(np.array([source_nodes, target_nodes]), dtype=torch.long)

# Edge Weights (For Node2Vec)
# We use log1p (log(1+x)) to shrink massive transactions so they don't break the random walker
edge_weights = torch.tensor(np.log1p(df['Amount Paid'].values), dtype=torch.float)

print("\n5. Building the PyG Data Object...")
graph_data = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_weights)

print("\n---- Final Upgraded Graph Data Object ----")
print(graph_data)
print(f"Contains Node Features? {graph_data.x is not None}")
print(f"Contains Edge Weights? {graph_data.edge_attr is not None}")


1. Loading Data & Mapping Nodes...
Total Unique Accounts (Nodes): 423676

2. Engineering Node Features (The 'Whale Crusher' Update)...

3. Aligning Features & Scaling for Neural Networks...
NEW Feature Matrix Shape: torch.Size([423676, 7]) (Should be [423676, 7])

4. Constructing Edges & Weights...

5. Building the PyG Data Object...

---- Final Upgraded Graph Data Object ----
Data(x=[423676, 7], edge_index=[2, 1000000], edge_attr=[1000000])
Contains Node Features? True
Contains Edge Weights? True


In [2]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling

print("\n--- PHASE 2: EXTRACTING NETWORK TOPOLOGY ---")
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

# 1. Force the GNN to learn PURE STRUCTURE (Dummy 1s)
structural_x = torch.ones((graph_data.num_nodes, 1), dtype=torch.float).to(device)

class StructuralSAGE(torch.nn.Module):
    def __init__(self):
        super(StructuralSAGE, self).__init__()
        self.conv1 = SAGEConv(1, 32) 
        self.conv2 = SAGEConv(32, 64)
        
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

model_struct = StructuralSAGE().to(device)
optimizer = torch.optim.Adam(model_struct.parameters(), lr=0.01)

def train_struct():
    model_struct.train()
    optimizer.zero_grad()
    z = model_struct(structural_x, graph_data.edge_index.to(device))
    
    pos_edge_index = graph_data.edge_index.to(device)
    pos_out = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=-1)
    
    neg_edge_index = negative_sampling(edge_index=pos_edge_index, num_nodes=z.size(0), num_neg_samples=pos_edge_index.size(1))
    neg_out = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=-1)

    loss = F.binary_cross_entropy_with_logits(pos_out, torch.ones_like(pos_out)) + \
           F.binary_cross_entropy_with_logits(neg_out, torch.zeros_like(neg_out))
    loss.backward()
    optimizer.step()

print("Training Structural-Only GraphSAGE...")
for epoch in range(1, 11): 
    train_struct()

# 2. Extract Embeddings
model_struct.eval()
with torch.no_grad():
    structural_embeddings = model_struct(structural_x, graph_data.edge_index.to(device)).cpu().numpy()

print("Structural Embeddings Ready!")
print(f"Shape: {structural_embeddings.shape}")


--- PHASE 2: EXTRACTING NETWORK TOPOLOGY ---
Training Structural-Only GraphSAGE...
Structural Embeddings Ready!
Shape: (423676, 64)


In [3]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_recall_curve, auc
import numpy as np

print("\n--- PHASE 3: THE 'SPLIT-BRAIN' ENSEMBLE SCORING ---")

# 1. Train Detector A: The Network Cop (Structure Only)
print("1. Scoring pure Network Structure (64D)...")
iso_struct = IsolationForest(n_estimators=100, random_state=42)
iso_struct.fit(structural_embeddings) 
continuous_struct_scores = -iso_struct.decision_function(structural_embeddings)

# 2. Train Detector B: The Financial Auditor (Money Only)
print("2. Scoring pure Financial Metadata (7D)...")
iso_fin = IsolationForest(n_estimators=100, random_state=42)
iso_fin.fit(x_tensor.numpy()) 
continuous_fin_scores = -iso_fin.decision_function(x_tensor.numpy())

# 3. Normalize the Scores
print("3. Normalizing and Ensembling Scores...")
scaler = MinMaxScaler()
struct_normalized = scaler.fit_transform(continuous_struct_scores.reshape(-1, 1)).flatten()
fin_normalized = scaler.fit_transform(continuous_fin_scores.reshape(-1, 1)).flatten()

# The Final Fusion: Add them together!
ensemble_risk_scores = struct_normalized + fin_normalized

# 4. Global Evaluation
df['Ensemble_Risk_Score'] = df['Sender_Node_ID'].map(lambda x: ensemble_risk_scores[x])

y_true = df['Is Laundering']
y_scores = df['Ensemble_Risk_Score']

precision_curve, recall_curve, thresholds = precision_recall_curve(y_true, y_scores, pos_label=1)
pr_auc = auc(recall_curve, precision_curve)
print(f"\n--- FINAL ENSEMBLE RESULTS ---")
print(f"Area Under the PR Curve (AUC-PR): {pr_auc:.4f}")

# 5. Bank Budgets
for budget in [1000, 5000, 10000]:
    dynamic_threshold = np.sort(y_scores)[-budget]
    df['Ensemble_Budget_Guess'] = (df['Ensemble_Risk_Score'] >= dynamic_threshold).astype(int)
    caught = df[(df['Is Laundering'] == 1) & (df['Ensemble_Budget_Guess'] == 1)].shape[0]
    print(f"Top {budget} Alerts: Caught {caught} criminals.")


--- PHASE 3: THE 'SPLIT-BRAIN' ENSEMBLE SCORING ---
1. Scoring pure Network Structure (64D)...
2. Scoring pure Financial Metadata (7D)...
3. Normalizing and Ensembling Scores...

--- FINAL ENSEMBLE RESULTS ---
Area Under the PR Curve (AUC-PR): 0.0064
Top 1000 Alerts: Caught 6 criminals.
Top 5000 Alerts: Caught 41 criminals.
Top 10000 Alerts: Caught 41 criminals.
